# `aidp-epm-cloud` live test — Identity-domain OAuth (v0.2)
**Live-test row 15.** Deferred to v0.2.


In [ ]:
import sys, os
# Adjust this if you've uploaded the plugin scripts/ dir elsewhere.
sys.path.insert(0, '/Workspace/Shared/oracle_ai_data_platform_connectors/scripts')


In [ ]:
from oracle_ai_data_platform_connectors.auth import oauth_token
import requests

token = oauth_token(
    token_url=os.environ['EPM_OAUTH_TOKEN_URL'],
    client_id=os.environ['EPM_OAUTH_CLIENT_ID'],
    private_key_pem=open(os.environ['EPM_OAUTH_PRIVATE_KEY_PATH']).read(),
)
session = requests.Session()
session.headers.update({'Authorization': f'Bearer {token}', 'Content-Type': 'application/json'})


In [ ]:
from oracle_ai_data_platform_connectors.rest.epm import (
    list_applications, export_data_slice, slice_to_long_dataframe,
)
apps = list_applications(session, os.environ['EPM_BASE_URL'])
print('apps:', [a['name'] for a in apps])
import json
grid = json.loads(os.environ['EPM_GRID_DEFINITION_JSON'])
resp = export_data_slice(session, os.environ['EPM_BASE_URL'], os.environ['EPM_APPLICATION'], os.environ['EPM_PLAN_TYPE'], grid)
df = slice_to_long_dataframe(spark, resp)
df.show(10)


In [ ]:
# Live-test driver parses this final cell's stdout for the JSON summary.
import json, time
summary = {
    'connector': 'aidp-epm-cloud',
    'auth': 'oauth',
    'rows': int(df.count()),
    'schema': sorted([f.name for f in df.schema.fields]),
    'timestamp_utc': int(time.time()),
}
print('AIDP_LIVE_TEST_RESULT_BEGIN')
print(json.dumps(summary, indent=2))
print('AIDP_LIVE_TEST_RESULT_END')
